# Predicting AQI - v1 (Lower MSE Focus)

This notebook is competition-platform friendly:
- Input: `dataset/public/train.csv`, `dataset/public/test.csv`, `dataset/public/sample_submission.csv`
- Output: `working/submission.csv`
- Strict time-aware CV (expanding window)
- Leakage-safe historical priors + target encoding
- Multi-model ensemble + OOF weight optimization


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PowerTransformer

try:
    from lightgbm import LGBMRegressor
except Exception as e:
    raise ImportError('lightgbm is required for this notebook.') from e

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


def locate_data_dir():
    candidates = [
        'dataset/public',
        './dataset/public',
        '/kaggle/input/predicting-air-quality-index',
        '/kaggle/input/predicting-air-quality-index-dataset',
        '/Users/songling/Desktop/Predicting Air Quality Index',
    ]
    for d in candidates:
        if all(os.path.exists(os.path.join(d, f)) for f in ['train.csv', 'test.csv', 'sample_submission.csv']):
            return d
    raise FileNotFoundError('Cannot locate train/test/sample_submission files.')


def make_ohe():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)


In [ ]:
DATA_DIR = locate_data_dir()
print('Using DATA_DIR:', DATA_DIR)

train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))

print('train:', train.shape, 'test:', test.shape)


In [ ]:
def base_feature_engineering(df):
    out = df.copy()
    out['date'] = pd.to_datetime(out['date'], errors='coerce')

    # explicit timestamp to keep chronology
    out['ts'] = pd.to_datetime(dict(year=out['year'], month=out['month'], day=out['day'])) + pd.to_timedelta(out['hour'], unit='h')
    out['date_ordinal'] = out['date'].map(lambda x: x.toordinal() if pd.notna(x) else np.nan)
    out['dayofyear'] = out['date'].dt.dayofyear.astype(float)
    out['weekofyear'] = out['date'].dt.isocalendar().week.astype(float)
    out['quarter'] = out['date'].dt.quarter.astype(float)

    # cyclical time features
    out['hour_sin'] = np.sin(2*np.pi*out['hour']/24.0)
    out['hour_cos'] = np.cos(2*np.pi*out['hour']/24.0)
    out['month_sin'] = np.sin(2*np.pi*out['month']/12.0)
    out['month_cos'] = np.cos(2*np.pi*out['month']/12.0)
    out['doy_sin'] = np.sin(2*np.pi*out['dayofyear']/365.25)
    out['doy_cos'] = np.cos(2*np.pi*out['dayofyear']/365.25)

    dow_map = {'Monday':0,'Tuesday':1,'Wednesday':2,'Thursday':3,'Friday':4,'Saturday':5,'Sunday':6}
    out['dow_num'] = out['day_of_week'].map(dow_map).fillna(0).astype(float)
    out['dow_sin'] = np.sin(2*np.pi*out['dow_num']/7.0)
    out['dow_cos'] = np.cos(2*np.pi*out['dow_num']/7.0)

    # interactions
    out['temp_humidity'] = out['temperature'] * out['humidity']
    out['temp_wind'] = out['temperature'] * out['wind_speed']
    out['hum_wind'] = out['humidity'] * out['wind_speed']
    out['wind_vis_ratio'] = out['wind_speed'] / (out['visibility'] + 1e-3)
    out['temp_hum_ratio'] = out['temperature'] / (out['humidity'] + 1e-3)
    out['visibility_inv'] = 1.0 / (out['visibility'] + 1e-3)
    out['is_night'] = out['hour'].isin([0,1,2,3,4,5,22,23]).astype(int)
    out['is_rush_hour'] = out['hour'].isin([7,8,9,17,18,19]).astype(int)

    # categorical crosses
    out['city_station'] = out['city'].astype(str) + '_' + out['station'].astype(str)
    out['city_hour'] = out['city'].astype(str) + '_' + out['hour'].astype(str)
    out['station_hour'] = out['station'].astype(str) + '_' + out['hour'].astype(str)
    out['station_month'] = out['station'].astype(str) + '_' + out['month'].astype(str)
    out['city_season'] = out['city'].astype(str) + '_' + out['season'].astype(str)

    return out


def add_history_features(train_df, test_df, target_col='aqi'):
    tr = train_df.copy().sort_values('ts').reset_index(drop=True)
    te = test_df.copy()

    global_mean = float(tr[target_col].mean())

    # keys for history priors
    keys = ['station', 'city', 'station_hour', 'city_hour', 'station_month', 'city_season']

    for k in keys:
        grp = tr.groupby(k)[target_col]
        csum = grp.cumsum() - tr[target_col]
        ccnt = grp.cumcount()

        prior = csum / ccnt.replace(0, np.nan)
        tr[f'prior_mean_{k}'] = prior.fillna(global_mean)
        tr[f'prior_count_{k}'] = ccnt.astype(float)

        # map train-full stats to test (future-safe)
        agg = tr.groupby(k)[target_col].agg(['mean', 'median', 'std', 'count'])
        te[f'prior_mean_{k}'] = te[k].map(agg['mean']).fillna(global_mean)
        te[f'prior_count_{k}'] = te[k].map(agg['count']).fillna(0).astype(float)
        te[f'prior_median_{k}'] = te[k].map(agg['median']).fillna(global_mean)
        te[f'prior_std_{k}'] = te[k].map(agg['std']).fillna(0.0)

    # global trend proxy by date_ordinal
    day_agg = tr.groupby('date_ordinal')[target_col].mean()
    tr['prior_day_mean'] = tr['date_ordinal'].map(day_agg).fillna(global_mean)
    te['prior_day_mean'] = te['date_ordinal'].map(day_agg).fillna(global_mean)

    # restore original order for train
    tr = tr.sort_values('id').reset_index(drop=True)
    return tr, te


def smooth_target_encode(ref_df, ref_y, apply_df, col, m=120):
    global_mean = float(ref_y.mean())
    tmp = ref_df[[col]].copy()
    tmp['_y'] = ref_y.values
    agg = tmp.groupby(col)['_y'].agg(['mean', 'count'])
    enc = (agg['mean'] * agg['count'] + global_mean * m) / (agg['count'] + m)
    return apply_df[col].map(enc).fillna(global_mean)


In [ ]:
# Build full feature tables
train_fe = base_feature_engineering(train)
test_fe = base_feature_engineering(test)

train_fe_hist, test_fe_hist = add_history_features(train_fe, test_fe, target_col='aqi')

# align train order to original id
train_fe_hist = train_fe_hist.sort_values('id').reset_index(drop=True)
test_fe_hist = test_fe_hist.sort_values('id').reset_index(drop=True)

# modeling tables
y = train_fe_hist['aqi'].astype(float)
X_base = train_fe_hist.drop(columns=['aqi', 'date', 'ts']).copy()
X_test_base = test_fe_hist.drop(columns=['date', 'ts']).copy()

# ensure identical columns (except id)
if 'id' in X_base.columns:
    X_base_noid = X_base.drop(columns=['id'])
else:
    X_base_noid = X_base.copy()
if 'id' in X_test_base.columns:
    X_test_noid = X_test_base.drop(columns=['id'])
else:
    X_test_noid = X_test_base.copy()

# chronological expanding folds
order = np.argsort(pd.to_datetime(train['date'], errors='coerce').values)
parts = np.array_split(order, 6)  # 5 validation folds
splits = []
for i in range(1, len(parts)):
    tr_idx = np.concatenate(parts[:i])
    va_idx = parts[i]
    if len(tr_idx) > 0 and len(va_idx) > 0:
        splits.append((tr_idx, va_idx))

print('Folds:', len(splits))
for i, (tr_idx, va_idx) in enumerate(splits, 1):
    print(f'Fold {i}: train={len(tr_idx)}, valid={len(va_idx)}')


In [ ]:
# model definitions
lgb_param_list = [
    dict(name='lgb_a', objective='regression', metric='l2', n_estimators=3000, learning_rate=0.015,
         num_leaves=224, min_child_samples=16, subsample=0.95, colsample_bytree=0.95,
         reg_alpha=0.05, reg_lambda=1.20, random_state=RANDOM_STATE, n_jobs=-1),
    dict(name='lgb_b', objective='regression', metric='l2', n_estimators=2600, learning_rate=0.018,
         num_leaves=192, min_child_samples=20, subsample=0.90, colsample_bytree=0.90,
         reg_alpha=0.15, reg_lambda=1.40, random_state=RANDOM_STATE+7, n_jobs=-1),
    dict(name='lgb_c', objective='regression', metric='l2', boosting_type='goss', n_estimators=2300, learning_rate=0.020,
         num_leaves=128, min_child_samples=25, subsample=1.0, colsample_bytree=0.85,
         reg_alpha=0.10, reg_lambda=1.80, random_state=RANDOM_STATE+13, n_jobs=-1),
]

cat_cols = ['day_of_week','season','city','station','city_station','city_hour','station_hour','station_month','city_season']
num_cols = [c for c in X_base_noid.columns if c not in cat_cols]

pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median'))]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', make_ohe())]), cat_cols)
], remainder='drop')

etr_template = TransformedTargetRegressor(
    regressor=Pipeline([
        ('pre', pre),
        ('reg', ExtraTreesRegressor(
            n_estimators=700,
            min_samples_leaf=2,
            max_features=0.85,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),
    transformer=PowerTransformer(method='yeo-johnson', standardize=True)
)

hgb_template = TransformedTargetRegressor(
    regressor=Pipeline([
        ('pre', pre),
        ('reg', HistGradientBoostingRegressor(
            learning_rate=0.03,
            max_depth=12,
            max_iter=900,
            min_samples_leaf=20,
            l2_regularization=0.15,
            random_state=RANDOM_STATE
        ))
    ]),
    transformer=PowerTransformer(method='yeo-johnson', standardize=True)
)


In [ ]:
# OOF + test predictions
model_oof = {}
model_test = {}

# helper: build lgb inputs with fold-safe target encoding
te_cols = ['city','station','season','day_of_week','city_station','city_hour','station_hour','station_month','city_season','hour','month']

def prepare_lgb(train_x, train_y, apply_x):
    tr = train_x.copy()
    ap = apply_x.copy()
    for c in te_cols:
        tr[f'te_{c}'] = smooth_target_encode(tr, train_y, tr, c, m=120)
        ap[f'te_{c}'] = smooth_target_encode(tr, train_y, ap, c, m=120)
    for c in cat_cols:
        tr[c] = tr[c].astype('category')
        ap[c] = ap[c].astype('category')
    return tr, ap

# prior-only baseline model (hierarchical weighted mean)
def prior_predict(ref_x, ref_y, query_x):
    gm = float(ref_y.mean())

    def map_stat(key, stat='mean'):
        agg = ref_x.join(ref_y.rename('aqi')).groupby(key)['aqi'].agg(['mean','count'])
        if stat == 'mean':
            return query_x[key].map(agg['mean']).fillna(gm), query_x[key].map(agg['count']).fillna(0)
        return query_x[key].map(agg['mean']).fillna(gm), query_x[key].map(agg['count']).fillna(0)

    p_station, c_station = map_stat('station')
    p_station_hour, c_station_hour = map_stat('station_hour')
    p_city_hour, c_city_hour = map_stat('city_hour')
    p_city, c_city = map_stat('city')

    # count-aware weights
    w1 = np.log1p(c_station_hour)
    w2 = np.log1p(c_station)
    w3 = np.log1p(c_city_hour)
    w4 = np.log1p(c_city)
    ws = w1 + w2 + w3 + w4 + 1e-9

    pred = (w1*p_station_hour + w2*p_station + w3*p_city_hour + w4*p_city) / ws
    return pred.values

# LGB models
for p in lgb_param_list:
    name = p['name']
    params = p.copy()
    params.pop('name')

    oof = np.zeros(len(X_base_noid), dtype=float)
    test_folds = []

    for tr_idx, va_idx in splits:
        Xtr = X_base_noid.iloc[tr_idx].copy()
        ytr = y.iloc[tr_idx].copy()
        Xva = X_base_noid.iloc[va_idx].copy()

        tr_m, va_m = prepare_lgb(Xtr, ytr, Xva)
        model = LGBMRegressor(**params)
        model.fit(tr_m, ytr)
        oof[va_idx] = model.predict(va_m)

        _, te_m = prepare_lgb(Xtr, ytr, X_test_noid.copy())
        test_folds.append(model.predict(te_m))

    model_oof[name] = oof
    model_test[name] = np.mean(np.vstack(test_folds), axis=0)

# ExtraTrees + HistGB
for name, template in [('etr', etr_template), ('hgb', hgb_template)]:
    oof = np.zeros(len(X_base_noid), dtype=float)
    test_folds = []
    for tr_idx, va_idx in splits:
        Xtr = X_base_noid.iloc[tr_idx].copy()
        ytr = y.iloc[tr_idx].copy()
        Xva = X_base_noid.iloc[va_idx].copy()

        model = template
        model.fit(Xtr, ytr)
        oof[va_idx] = model.predict(Xva)
        test_folds.append(model.predict(X_test_noid))

    model_oof[name] = oof
    model_test[name] = np.mean(np.vstack(test_folds), axis=0)

# Prior baseline as one more robust model
prior_oof = np.zeros(len(X_base_noid), dtype=float)
prior_test_folds = []
for tr_idx, va_idx in splits:
    Xtr = X_base_noid.iloc[tr_idx].copy()
    ytr = y.iloc[tr_idx].copy()
    Xva = X_base_noid.iloc[va_idx].copy()

    prior_oof[va_idx] = prior_predict(Xtr, ytr, Xva)
    prior_test_folds.append(prior_predict(Xtr, ytr, X_test_noid))

model_oof['prior'] = prior_oof
model_test['prior'] = np.mean(np.vstack(prior_test_folds), axis=0)

for k, p in model_oof.items():
    mask = p != 0
    mse = mean_squared_error(y[mask], p[mask])
    print(f'{k} OOF MSE: {mse:.6f}')


In [ ]:
# OOF weight optimization + robust post-processing
names = list(model_oof.keys())
P = np.column_stack([model_oof[n] for n in names])
Pt = np.column_stack([model_test[n] for n in names])

valid_mask = np.any(P != 0, axis=1)
Pv = P[valid_mask]
yv = y.values[valid_mask]

best_mse = 1e18
best_w = None
rng = np.random.default_rng(RANDOM_STATE)

# random simplex search
for _ in range(120000):
    w = rng.dirichlet(np.ones(len(names)) * 1.2)
    pred = Pv @ w
    mse = mean_squared_error(yv, pred)
    if mse < best_mse:
        best_mse = mse
        best_w = w

print('Best OOF MSE:', best_mse)
print('Best weights:', {n: float(w) for n, w in zip(names, best_w)})

# final blend
pred_test = Pt @ best_w

# conservative outlier control (keeps ranking stable)
low_q, high_q = np.quantile(y, [0.003, 0.997])
pred_test = np.clip(pred_test, low_q, high_q)

# hard task bounds
pred_test = np.clip(pred_test, 25.0, 500.0)

submission = pd.DataFrame({
    'id': test['id'].astype(int),
    'aqi': pred_test
}).sort_values('id').reset_index(drop=True)

assert len(submission) == len(test)
os.makedirs('working', exist_ok=True)
out_path = 'working/submission.csv'
submission.to_csv(out_path, index=False)

print('Saved:', out_path)
print('Shape:', submission.shape)
print(submission.head())
